In [1]:
#using uproot to write generated event into TTree

import uproot

import numpy as np
import pandas as pd

import naive



/home/atam/hackjet/naive/naive.py:373: SyntaxWarning: invalid escape sequence '\m'
  """


In [2]:
#fastjet version of naive.GhostJetPopulator

def GhostJetPopulator_fastjet_version(jet_input, y_min, y_max, ny, phi_min, phi_max, nphi, epsil=1e-6, version=1):
  """
  Populates a set of input jets with ghosts jets. In particular, takes jet_input set,
  and adds a series of ny*nphi many ghost jets in a uniform (y,phi) grid.

  Parameters:
      jet_input (list): List of input jets.
      y_min, y_max (float): rapidity (y) bounds of the uniform (y,phi) grid where ghost jets are populated.
      phi_min, phi_max (float): phi-bounds of the uniform (y,phi) grid where ghost jets are populated.
      ny, nphi: number of ghost jets populated along y,phi axis respectively, resulting in N = ny*nphi many ghost jets.

      epsil (float): Energy of ghost jet, by default a soft, massless jet, with E = |p|= epsil.

      version (-1 or 1): alternating between versions of ghost jet indexing.

  Returns:
      list: List of input jets with added ghost jets.

  """

  y_axis = np.linspace(y_min, y_max, ny)
  phi_axis = np.linspace(phi_min, phi_max, nphi)


  y_grid, phi_grid = np.meshgrid(y_axis, phi_axis)

  y_phi_coordinates = np.column_stack((y_grid.ravel(), phi_grid.ravel()))

  # creating soft jets, using E = |p|= epsil,
  # and obtaining p-components through y, phi with
  # pz = E*tanh(y); pT = E**2 - pz**2,
  # px = pT*cos(phi); py = pT*sin(phi)

  pz_values = epsil * np.tanh(y_grid.ravel())
  pT_values = np.sqrt(epsil**2 - pz_values**2)
  px_values = pT_values * np.cos(phi_grid.ravel())
  py_values = pT_values * np.sin(phi_grid.ravel())

  #assigns indices to ghost jets sequentially, as len(jet_input), len(jet_input)+1, len(jet_input)+2, and so on.
  #input particles in jet_input assumed to be indexed as integers, from 0, 1, 2, ..., len(jet_input)-1.

  if version == 1:
    ghost_jets = [fastjet.PseudoJet(px, py, pz, E) for i, (px, py, pz) in enumerate(zip(px_values.ravel(), py_values.ravel(), pz_values.ravel()))]
  elif version == -1:
    ghost_jets = [fastjet.PseudoJet(px, py, pz, E) for i, (px, py, pz) in enumerate(zip(px_values.ravel(), py_values.ravel(), pz_values.ravel()))]
  else:
    raise ValueError("Invalid version value. Must be -1 or 1.")
  #alternate possibility: index as negative numbers [i+len(jet_input)] --> [-(i+1)] (as i starts at 0), so one can more easily distinguish ghost jets by index


  jet_input += ghost_jets

  return jet_input

  jet_input += ghost_jets

  return jet_input



In [3]:
#read event data

file4 = uproot.open("PythiaEventsBatchTest.root")

tree = file4['pdEventTree']

#read branches into Awkward array, then numpy arrays

E_array = tree['E'].array().to_numpy()
px_array = tree['px'].array().to_numpy()
py_array = tree['py'].array().to_numpy()
pz_array = tree['pz'].array().to_numpy()
eta_array = tree['eta'].array().to_numpy()
phi_array = tree['phi'].array().to_numpy()



In [4]:
# using FastJet now:
import fastjet

In [5]:
#Bringing in FastJet's anti-kT, creating PseudoJets from input data
#Can make minor edits to use CA, kT algorithms as well

antikt_jetdef = fastjet.JetDefinition(fastjet.antikt_algorithm, 0.4)

kt_jetdef = fastjet.JetDefinition(fastjet.kt_algorithm, 0.4)

ca_jetdef = fastjet.JetDefinition(fastjet.cambridge_algorithm, 0.4)

jet_set = [fastjet.PseudoJet(px_array[i], py_array[i], pz_array[i], E_array[i]) for i in range(len(E_array))]



In [6]:
for i,j in enumerate(jet_set):
    j.set_user_index(i)

In [7]:
#Clustering with no ghost jets

antikTcluster = fastjet.ClusterSequence(jet_set, antikt_jetdef)

ktcluster = fastjet.ClusterSequence(jet_set, kt_jetdef)

cacluster = fastjet.ClusterSequence(jet_set, ca_jetdef)



#--------------------------------------------------------------------------
#                         FastJet release 3.5.1
#                 M. Cacciari, G.P. Salam and G. Soyez                  
#     A software package for jet finding and analysis at colliders      
#                           https://fastjet.fr                           
#	                                                                      
# Please cite EPJC72(2012)1896 [arXiv:1111.6097] if you use this package
# for scientific work and optionally PLB641(2006)57 [hep-ph/0512210].   
#                                                                       
# FastJet is provided without warranty under the GNU GPL v2 or higher.  
# It uses T. Chan's closest pair algorithm, S. Fortune's Voronoi code,
# CGAL and 3rd party plugin jet algorithms. See COPYING file for details.
#--------------------------------------------------------------------------


In [8]:
import time
import statistics

def TimingMeasurement(inputjets, ClusterFunc, *args, **kwargs):

  """
  Measures how much time it takes to cluster a set of jets.

  returns N, time_result, dt
    with N = number of initial jets clustered (including ghosts),
    time_result = average time elapsed in seconds,
    and dt = standard deviation from five trials

  """

  t_list = []

  for count in range(5):

    print(f"Trial {count} begins...")
    start = time.perf_counter()
    ClusterFunc(inputjets, *args, **kwargs)
    end = time.perf_counter()

    t_list.append(end-start)
    print(f"Trial {count} complete: reading {end-start} seconds")

  dt = statistics.stdev(t_list)
  time_result = statistics.mean(t_list)



  return len(inputjets), time_result, dt

In [9]:
InputJets = [fastjet.PseudoJet(px_array[i], py_array[i], pz_array[i], E_array[i]) for i in range(len(E_array))]

In [10]:
pi = float(np.pi)

In [11]:
def fj_ghost_jets(inputjet_set, y_min, y_max, n_y, phi_min, phi_max, n_phi, epsil=1e-6):
  y_axis = np.linspace(y_min, y_max, n_y)
  phi_axis = np.linspace(phi_min, phi_max, n_phi)

  y_grid, phi_grid = np.meshgrid(y_axis, phi_axis)

  y_phi_coordinates = np.column_stack((y_grid.ravel(), phi_grid.ravel()))

  pz_values = epsil * np.tanh(y_grid.ravel())
  pT_values = np.sqrt(epsil**2 - pz_values**2)
  px_values = pT_values * np.cos(phi_grid.ravel())
  py_values = pT_values * np.sin(phi_grid.ravel())

  ghost_jets = [fastjet.PseudoJet(px_values[i], py_values[i], pz_values[i], epsil) for i in range(len(pz_values))]

  inputjet_set += ghost_jets

  for i,j in enumerate(inputjet_set):
    j.set_user_index(i)



  return inputjet_set

In [12]:
nlist = [5,6,7,8,9,10]

N_data = []
t_data = []
dt_data = []

runcount = len(nlist)

#just comparing antikt:

for _n in nlist:

  InputJetsGhosted = fj_ghost_jets(inputjet_set=InputJets, y_min=-1.0, y_max=1.0, n_y=_n, phi_min=0.0, phi_max=2*pi, n_phi=_n)

  n_result, t_result, dt_result = TimingMeasurement(InputJetsGhosted, fastjet.ClusterSequence, antikt_jetdef)

  N_data.append(n_result)
  t_data.append(t_result)
  dt_data.append(dt_result)

  print(f"{n_result} clustered in {t_result} +/- {dt_result} seconds")

Trial 0 begins...
Trial 0 complete: reading 0.0002665149995664251 seconds
Trial 1 begins...
Trial 1 complete: reading 0.0001665090003371006 seconds
Trial 2 begins...
Trial 2 complete: reading 0.00014357300005940488 seconds
Trial 3 begins...
Trial 3 complete: reading 0.0002764999999271822 seconds
Trial 4 begins...
Trial 4 complete: reading 0.00016047799999796553 seconds
168 clustered in 0.00020271499997761566 +/- 6.345723145683479e-05 seconds
Trial 0 begins...
Trial 0 complete: reading 0.00019929900008719414 seconds
Trial 1 begins...
Trial 1 complete: reading 0.00018046200057142414 seconds
Trial 2 begins...
Trial 2 complete: reading 0.00017366100018989528 seconds
Trial 3 begins...
Trial 3 complete: reading 0.00017219199980900157 seconds
Trial 4 begins...
Trial 4 complete: reading 0.00018441100019117584 seconds
204 clustered in 0.0001820050001697382 +/- 1.0876228996669267e-05 seconds
Trial 0 begins...
Trial 0 complete: reading 0.00028666399975918466 seconds
Trial 1 begins...
Trial 1 comp

In [13]:
tvsNdata = pd.DataFrame({
    "N": N_data,
    "t": t_data,
    "dt": dt_data
})

In [14]:
tvsNdata.to_csv('fj_timing_data.csv')